# Lighting Persona - Transformer

Trains the local Transformer lighting persona model and saves reports/artifacts to Drive.

Artifacts are saved back to Google Drive under `AIModelsAndAlgorithms/LightingPersona/transformer`.


In [8]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

# 1) Mount Google Drive for datasets and saved training outputs.
drive.mount('/content/drive')

# 2) Edit these two values for your GitHub repo.
GITHUB_REPO_URL = 'https://github.com/abdullahtapanci/VisualizationApp.git'
GITHUB_BRANCH = 'main'  # change if you use another branch

# 3) Drive folders. Datasets stay in Drive; code is cloned from GitHub.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/VisualizationApp')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'Data'
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_DIR
PROJECT_DIR = Path('/content/VisualizationApp')

if 'YOUR_USERNAME' in GITHUB_REPO_URL:
    raise ValueError('Edit GITHUB_REPO_URL to your real GitHub repository URL, then rerun this cell.')
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f'Drive dataset folder was not found: {DRIVE_DATA_DIR}')

# 4) Clone or update code from GitHub.
if PROJECT_DIR.exists():
    print('Repository already exists. Pulling latest changes...')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)

# 5) Link Drive datasets into the cloned repo so existing scripts read Data/*.csv.
LOCAL_DATA_DIR = PROJECT_DIR / 'Data'
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
    src = DRIVE_DATA_DIR / filename
    dst = LOCAL_DATA_DIR / filename
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('linked', dst, '->', src)
    else:
        print('not found in Drive, skipping:', src)

DATA_DIR = LOCAL_DATA_DIR
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('DRIVE_OUTPUT_ROOT =', DRIVE_OUTPUT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository already exists. Pulling latest changes...
linked /content/VisualizationApp/Data/PIRSensorData.csv -> /content/drive/MyDrive/VisualizationApp/Data/PIRSensorData.csv
linked /content/VisualizationApp/Data/hotelReservationData.csv -> /content/drive/MyDrive/VisualizationApp/Data/hotelReservationData.csv
linked /content/VisualizationApp/Data/lightningData.csv -> /content/drive/MyDrive/VisualizationApp/Data/lightningData.csv
linked /content/VisualizationApp/Data/temperatureData.csv -> /content/drive/MyDrive/VisualizationApp/Data/temperatureData.csv
linked /content/VisualizationApp/Data/WheatherDataAntalya.csv -> /content/drive/MyDrive/VisualizationApp/Data/WheatherDataAntalya.csv
PROJECT_DIR = /content/VisualizationApp
DATA_DIR = /content/VisualizationApp/Data
DRIVE_OUTPUT_ROOT = /content/drive/MyDrive/VisualizationApp


In [9]:
# Optional: if your new dataset files are in another Drive folder, set NEW_DATA_DIR and run this cell.
# The files will be linked/copied through DRIVE_DATA_DIR and then used by the cloned repo.
# Example: NEW_DATA_DIR = Path('/content/drive/MyDrive/NewVisualizationAppData')
NEW_DATA_DIR = None

if NEW_DATA_DIR is not None:
    for filename in ['PIRSensorData.csv', 'hotelReservationData.csv', 'lightningData.csv', 'temperatureData.csv', 'WheatherDataAntalya.csv']:
        src = Path(NEW_DATA_DIR) / filename
        dst = DRIVE_DATA_DIR / filename
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            !cp "{src}" "{dst}"
            print('copied', src, '->', dst)
        else:
            print('missing in NEW_DATA_DIR:', src)


In [10]:
!pip install -q pandas numpy scikit-learn joblib matplotlib torch


In [11]:
assert (DATA_DIR / 'lightningData.csv').exists(), 'Missing lightningData.csv'
OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'AIModelsAndAlgorithms/LightingPersona/transformer'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)


OUTPUT_DIR = /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingPersona/transformer


## Smoke Test
Run this first to verify paths and dependencies without training on the full dataset.


In [12]:
%cd {PROJECT_DIR}
!python AIModelsAndAlgorithms/LightingPersona/transformer/lighting_persona_transformer.py --max-rows 500000 --max-sequences 50000 --sequence-length 24 --stride 12 --epochs 2 --batch-size 128 --output-dir "{OUTPUT_DIR}"


/content/VisualizationApp
/content/VisualizationApp/AIModelsAndAlgorithms/LightingPersona/transformer/lighting_persona_transformer.py:183: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
epoch 001 train_loss=1.5253 test_acc=0.500
epoch 002 train_loss=1.2339 test_acc=0.461
samples: 3,468
train samples: 2,774
test samples: 694
sequence length: 24
stride: 12
input features: 19
time range: 2022-01-06 22:00:00 to 2022-02-05 23:55:00
best test accuracy: 0.500

              precision    recall  f1-score   support

    Balanced      0.570     0.343     0.429       166
NightFocused      0.019     0.022     0.020        90
     Routine      0.813     0.884     0.847       207
StaticBright      0.272     0.319     0.293        69
   StaticDim      0.461     0.512     0.485       162

    accuracy                          0.500       694
   macro avg 

## Full Training
Run this cell when you are ready. It may take a long time on the full dataset.


In [13]:
%cd {PROJECT_DIR}
!python AIModelsAndAlgorithms/LightingPersona/transformer/lighting_persona_transformer.py --max-sequences 300000 --sequence-length 48 --stride 3 --epochs 30 --batch-size 256 --output-dir "{OUTPUT_DIR}"


/content/VisualizationApp
^C


## Check Saved Artifacts


In [14]:
print('Saved files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)


Saved files:
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingPersona/transformer/lighting_persona_transformer.pt
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingPersona/transformer/lighting_persona_transformer_confusion_matrix.csv
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingPersona/transformer/lighting_persona_transformer_metadata.json
/content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/LightingPersona/transformer/lighting_persona_transformer_report.txt
